In [1]:
!pip install faker -q

import pandas as pd
import numpy as np
from faker import Faker
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder # dar uma olhada
from sklearn.compose import ColumnTransformer   # olhar
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline # olhar
import warnings
import os

In [ ]:
caminho_original = '../data/cardio_train.csv'
caminho_personal = '../data/personal_cardio_train.csv'

if not os.path.exists(caminho_original):
    caminho_original = 'cardio_train.csv'
    caminho_personal = 'personal_cardio_train.csv'


try:
    df_real = pd.read_csv(caminho_original, sep=';')
        
    print(f"Dataset carregado: {df_real.shape[0]} linhas e {df_real.shape[1]} colunas.")
    

    if 'age' in df_real.columns:
        df_real['age'] = (df_real['age'] / 365.25).astype(int)
        print("Transformação: Coluna 'age' convertida de dias para anos.")
    

    fake = Faker('pt_BR')
    print("A injetar Nomes e CPFs fictícios (Mock de Dados Pessoais)")
    
    if 'nome' not in df_real.columns:
        nomes_por_genero = []
        for g in df_real['gender']:
            if g == 1:
                nomes_por_genero.append(fake.name_female())
            elif g == 2:
                nomes_por_genero.append(fake.name_male())
            else:
                nomes_por_genero.append(fake.name())
                
        df_real.insert(0, 'nome', nomes_por_genero)
    
    if 'cpf' not in df_real.columns:
        df_real.insert(1, 'cpf', [fake.ssn() for i in range(len(df_real))])


    if os.path.dirname(caminho_personal):
        os.makedirs(os.path.dirname(caminho_personal), exist_ok=True)
    df_real.to_csv(caminho_personal, index=False, sep=';')
    
    print(f"Arquivo processado e salvo em: {caminho_personal}\n")
    display(df_real[['nome', 'cpf', 'id', 'age', 'gender', 'height',
                     'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc',
                     'smoke', 'alco', 'cardio']].head(3))

except FileNotFoundError:
    print(f"O arquivo {caminho_original} não foi encontrado.")

O arquivo ../cardio_train.csv não foi encontrado.


In [6]:
caminho_anonimizado = '../data/anonimized_cardio_train.csv' if caminho_personal.startswith('../') else 'anonimized_cardio_train.csv'


df_pessoal = pd.read_csv(caminho_personal, sep=';')
df_anon = df_pessoal.copy()


k_alvo = 1500
print(f"Aplicar K-Anonimato (k={k_alvo}).")

# O K-Anonimato suprime identificadores diretos
df_anon = df_anon.drop(columns=['nome', 'cpf', 'id'])

# Generalizar atributos quase-identificadores (QI)
df_anon['age_group'] = pd.cut(df_anon['age'], bins=[0, 35, 50, 65, 120], labels=['Jovem', 'Adulto', 'Meia-Idade', 'Sênior'])
df_anon['weight_group'] = pd.cut(df_anon['weight'], bins=[0, 50, 65, 80, 95, 200], labels=['<50kg', '50-65kg', '66-80kg', '81-95kg', '>95kg'])
df_anon['height_group'] = pd.cut(df_anon['height'], bins=[0, 150, 160, 170, 180, 250], labels=['<150cm', '150-160cm', '161-170cm', '171-180cm', '>180cm'])

df_anon = df_anon.drop(columns=['age', 'weight', 'height'])

# Suprimir registros anonimizados não diferenciáveis de k outros registros (Outliers)
# Garantia de K Grupos
quase_identificadores_finais = ['age_group', 'gender', 'weight_group', 'height_group']
contagem = df_anon.groupby(quase_identificadores_finais, observed=False).size().reset_index(name='Tamanho')
df_k_anon = df_anon.merge(contagem, on=quase_identificadores_finais)

df_final = df_k_anon[df_k_anon['Tamanho'] >= k_alvo].copy()
df_final = df_final.drop(columns=['Tamanho'])


df_final.to_csv(caminho_anonimizado, index=False, sep=';')
print(f"Arquivo K-Anonimizado salvo em: {caminho_anonimizado}\n")
display(df_final.head(3))

Aplicar K-Anonimato (k=1500).
Arquivo K-Anonimizado salvo em: anonimized_cardio_train.csv



,gender,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_group,weight_group,height_group
1103,1,140,90,3,1,0,0,1,1,Meia-Idade,81-95kg,150-160cm
1104,1,130,80,3,1,0,0,1,0,Meia-Idade,81-95kg,150-160cm
1105,1,145,85,2,2,0,0,1,1,Meia-Idade,81-95kg,150-160cm


In [10]:
l_alvo = 1

colunas_sensiveis = ['cardio', 'cholesterol', 'gluc']

df_l_diversidade = df_final.copy() 

print(f"Aplicar L-Diversidade (l={l_alvo}) nos múltiplos atributos: {colunas_sensiveis}.")

df_l_diversidade['comb_sensivel'] = df_l_diversidade[colunas_sensiveis].astype(str).agg('_'.join, axis=1)

# Agrupar pelos quase-identificadores e contar as combinações ÚNICAS da coluna composta
diversidade = df_l_diversidade.groupby(quase_identificadores_finais, observed=False)['comb_sensivel'].nunique().reset_index(name='l_valores_distintos')

# Mesclar com o dataset k-anonimizado
df_l_div = df_l_diversidade.merge(diversidade, on=quase_identificadores_finais)

# Filtrar para manter APENAS os grupos que atendem à regra de l-diversidade
df_l_diversidade = df_l_div[df_l_div['l_valores_distintos'] >= l_alvo].copy()

df_l_diversidade = df_l_diversidade.drop(columns=['comb_sensivel', 'l_valores_distintos'])

registros_perdidos = len(df_final) - len(df_l_diversidade)
print(f"-> Registros mantidos após L-Diversidade: {len(df_l_diversidade)}")
print(f"-> Registros suprimidos pela L-Diversidade: {registros_perdidos}")

df_l_diversidade.to_csv(caminho_anonimizado, index=False, sep=';')
df_final = df_l_diversidade
display(df_final.head(3))

Aplicar L-Diversidade (l=1) nos múltiplos atributos: ['cardio', 'cholesterol', 'gluc'].
-> Registros mantidos após L-Diversidade: 69978
-> Registros suprimidos pela L-Diversidade: 0


,gender,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_group,weight_group,height_group
0,2,110,80,1,1,0,0,1,0,Adulto,50-65kg,161-170cm
1,2,120,80,1,1,0,0,0,0,Adulto,50-65kg,161-170cm
2,2,120,80,2,1,0,0,1,1,Adulto,50-65kg,161-170cm


In [11]:
t_alvo = 0.20
coluna_sensivel_t = 'cardio'

print(f"Aplicar T-Proximidade (t={t_alvo}) no atributo sensível: '{coluna_sensivel_t}'.")

df_t_proximidade = df_final.copy()

prob_global = df_t_proximidade[coluna_sensivel_t].value_counts(normalize=True).to_dict()

grupos = df_t_proximidade.groupby(quase_identificadores_finais, observed=False)
grupos_validos = []

for nome_grupo, grupo in grupos:
    if len(grupo) == 0:
        continue
        
    prob_local = grupo[coluna_sensivel_t].value_counts(normalize=True).to_dict()
    
    categorias = set(prob_global.keys()).union(set(prob_local.keys()))
    distancia = 0.5 * sum(abs(prob_local.get(c, 0) - prob_global.get(c, 0)) for c in categorias)
    
    
    if distancia <= t_alvo:
        grupos_validos.append(nome_grupo)


chaves_grupos = df_t_proximidade.set_index(quase_identificadores_finais).index
df_t_proximidade = df_t_proximidade[chaves_grupos.isin(grupos_validos)].copy()

registros_perdidos_t = len(df_final) - len(df_t_proximidade)
print(f"-> Registros mantidos após T-Proximidade: {len(df_t_proximidade)}")
print(f"-> Registros suprimidos pela T-Proximidade: {registros_perdidos_t}")


df_t_proximidade.to_csv(caminho_anonimizado, index=False, sep=';')
df_final = df_t_proximidade
display(df_final.head(3))

Aplicar T-Proximidade (t=0.2) no atributo sensível: 'cardio'.
-> Registros mantidos após T-Proximidade: 58595
-> Registros suprimidos pela T-Proximidade: 11383


,gender,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_group,weight_group,height_group
1103,1,140,90,3,1,0,0,1,1,Meia-Idade,81-95kg,150-160cm
1104,1,130,80,3,1,0,0,1,0,Meia-Idade,81-95kg,150-160cm
1105,1,145,85,2,2,0,0,1,1,Meia-Idade,81-95kg,150-160cm


In [7]:
coluna_alvo = 'cardio'
colunas_medicas = ['ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']

def treinar_modelo(df_treino, features, alvo):
    X = df_treino[features]
    y = df_treino[alvo]
    
    # Estudar
    cols_categoricas = X.select_dtypes(include=['object', 'category']).columns.tolist()
    preprocessor = ColumnTransformer(
        transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), cols_categoricas)], 
        remainder='passthrough'
    )
    
    X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Estudar random_state
    modelo = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10))
    ])
    
    modelo.fit(X_treino, y_treino)
    return accuracy_score(y_teste, modelo.predict(X_teste)) # considerar retornar loss tbm

print("[1/2] Treinar IA sem anonimizar.")
features_baseline = ['age', 'gender', 'height', 'weight'] + colunas_medicas
acuracia_baseline = treinar_modelo(df_pessoal, features_baseline, coluna_alvo)

print("[2/2] Treinar IA com dados anonimizados.")
features_anon = quase_identificadores_finais + colunas_medicas
acuracia_anon = treinar_modelo(df_final, features_anon, coluna_alvo)


print("\n" + "="*60)
print(" PoC1 - RANDOM-FOREST COM DADOS ANONIMIZADOS")
print("="*60)
print(f"Total de Registros (Baseline): {len(df_pessoal)}")
print(f"Registros Mantidos (K={k_alvo}): {len(df_final)}")
print(f"Registros Suprimidos (Outliers): {len(df_pessoal) - len(df_final)} {100.0-(len(df_final)*100)/len(df_pessoal):.2f}%")
print("-" * 60)
print(f"Acurácia Baseline (Sem Privacidade):   {acuracia_baseline*100:.2f}%")
print(f"Acurácia K-Anonimato (Com Privacidade): {acuracia_anon*100:.2f}%")
print(f"Custo de Privacidade (Queda):          {(acuracia_baseline - acuracia_anon)*100:.2f} pontos percentuais")
print("="*60)

[1/2] Treinar IA sem anonimizar.
[2/2] Treinar IA com dados anonimizados.

 PoC1 - RANDOM-FOREST COM DADOS ANONIMIZADOS
Total de Registros (Baseline): 70000
Registros Mantidos (K=1500): 50078
Registros Suprimidos (Outliers): 19922 28.46%
------------------------------------------------------------
Acurácia Baseline (Sem Privacidade):   74.01%
Acurácia K-Anonimato (Com Privacidade): 72.20%
Custo de Privacidade (Queda):          1.80 pontos percentuais
